# La Firma Sin Sentido: Falsificando ECDSA Sin Clave Privada

**Based on:** [Faketoshi's Nonsense Signature](https://jimmysong.medium.com/faketoshis-nonsense-signature-8700a44536b5) by Jimmy Song (2018)  
**Math credit:** Andy Poelstra, Greg Maxwell, Pieter Wuille

[← Back to ECC Teachable Scheme](./00-ecc-teachable-scheme.ipynb) | [El Truco de Wright (2016) →](./01-ecc-wright-trick.ipynb)

---

## The Setup

In November 2018, a Twitter account posted a "signature" from Satoshi's
genesis block public key. It verified. People panicked.

But the signature was worthless — a mathematical parlor trick that
**anyone** can do for **any** public key, without knowing the private key.

This is a different attack than the [2016 Wright blog trick](./01-ecc-wright-trick.ipynb)
(which replayed a real signature from the blockchain). This one is pure algebra —
and it teaches a deeper lesson about what ECDSA actually guarantees.

### Jimmy Song's analogy

> It's like "proving" you ran a marathon in under 2 hours by starting
> near the finish line.

---

## Parte 1: Cómo Funciona Realmente la Verificación ECDSA

Before we can understand the trick, we need to see verification as an equation
with variables — not just a black box that says "valid" or "invalid."

### The verification formula

Given public key $P$, message hash $z$, and signature $(r, s)$:

$$u = z \cdot s^{-1} \bmod N$$
$$v = r \cdot s^{-1} \bmod N$$
$$R' = u \times G + v \times P$$
$$\text{Valid if } R'_x \equiv r \pmod{N}$$

### Contar los grados de libertad

```
Normal signing:                      The trick:
──────────────                       ──────────
GIVEN: z (hash of real message)      GIVEN: P (public key — from blockchain)
GIVEN: P (public key)                CHOOSE: u, v (random)
SECRET: d (private key)              DERIVE: r, s, z (to satisfy the formula)
OUTPUT: (r, s)

The signer has 1 free variable (k)   The attacker has 2 free variables (u, v)
and MUST satisfy z.                  and gets to PICK z.

More freedom → can satisfy the       But z is garbage, not a hash of
equation without knowing d.          any real message.
```

### La observación de Pieter Wuille

> "ECDSA signatures where the message isn't a hash and chosen by the
> 'signer' are insecure." — Pieter Wuille

The entire security of ECDSA rests on the signer NOT controlling $z$.
The moment they do, the private key becomes irrelevant.

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import hashlib
import secrets

from ecc import *

def ecdsa_verify_z(z: int, r: int, s: int, pub: Point) -> bool:
    s_inv = pow(s, SECP_N - 2, SECP_N)
    u = (z * s_inv) % SECP_N
    v = (r * s_inv) % SECP_N
    R = point_add(scalar_mult(u, G), scalar_mult(v, pub))
    return R.x % SECP_N == r

print("Primitivas criptográficas cargadas.")

## Parte 2: La Falsificación — Paso a Paso

We'll forge a "valid" ECDSA signature for Satoshi's genesis block public key.
We don't know the private key. We never will. But the signature will verify.

### El algoritmo

```
1. Pick random u, v
2. Compute R = u×G + v×P         (P is the target public key)
3. Set r = R.x mod N
4. Set s = r / v mod N            (from verification formula: v = r/s)
5. Set z = u × s mod N            (from verification formula: u = z/s)
6. The "signature" is (r, s) for "message hash" z
```

### ¿Por qué pasa la verificación?

The verifier computes:
- $u' = z \cdot s^{-1} = (u \cdot s) \cdot s^{-1} = u$ 
- $v' = r \cdot s^{-1} = r \cdot (v / r) = v$
- $R' = u' \times G + v' \times P = u \times G + v \times P = R$
- $R'_x = R_x = r$ ✓

We engineered it so the verification equation is trivially satisfied.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  LA FALSIFICACIÓN
# ═══════════════════════════════════════════════════════════════

# Target: Satoshi's genesis block public key (anyone can read this)
GENESIS_PUBKEY_HEX = (
    "04678afdb0fe5548271967f1a67130b7105cd6a828e03909"
    "a67962e0ea1f61deb649f6bc3f4cef38c4f35504e51ec112"
    "de5c384df7ba0b8d578a4c702b6bf11d5f"
)

# Parse the uncompressed public key (04 || x || y)
pubkey_bytes = bytes.fromhex(GENESIS_PUBKEY_HEX)
px = int.from_bytes(pubkey_bytes[1:33], 'big')
py = int.from_bytes(pubkey_bytes[33:65], 'big')
P = Point(px, py)

# Verify P is on the curve
assert (py * py) % SECP_P == (px * px * px + 7) % SECP_P
print("Target: Satoshi's genesis block public key")
print(f"  P.x = {hex(px)[:24]}...")
print(f"  P.y = {hex(py)[:24]}...")
print(f"  On curve: True")
print(f"\nWe do NOT know the private key.")
print(f"We are about to forge a 'valid' signature anyway.")

In [ ]:
# Step 1: Pick random u and v
u = secrets.randbelow(SECP_N - 1) + 1
v = secrets.randbelow(SECP_N - 1) + 1

print("STEP 1: Pick random u, v")
print(f"  u = {hex(u)[:20]}...  (random)")
print(f"  v = {hex(v)[:20]}...  (random)")

# Step 2: Compute R = u×G + v×P
R = point_add(scalar_mult(u, G), scalar_mult(v, P))
print(f"\nSTEP 2: R = u×G + v×P")
print(f"  R.x = {hex(R.x)[:24]}...")

# Step 3: r = R.x mod N
r = R.x % SECP_N
print(f"\nSTEP 3: r = R.x mod N")
print(f"  r = {hex(r)[:24]}...")

# Step 4: s = r/v mod N  (from v = r/s → s = r/v)
v_inv = pow(v, SECP_N - 2, SECP_N)
s = (r * v_inv) % SECP_N
print(f"\nSTEP 4: s = r × v⁻¹ mod N")
print(f"  s = {hex(s)[:24]}...")

# Step 5: z = u×s mod N  (from u = z/s → z = u×s)
z = (u * s) % SECP_N
print(f"\nSTEP 5: z = u × s mod N")
print(f"  z = {hex(z)[:24]}...")
print(f"  (This z is GARBAGE — it's not a hash of any real message)")

In [ ]:
# Step 6: Verify — does it pass?
print("STEP 6: Verification")
print("=" * 50)

valid = ecdsa_verify_z(z, r, s, P)

print(f"  ecdsa_verify_z(z, r, s, P) = {valid}")
print(f"")
print(f"  The signature VERIFIES. ✓")
print(f"  For Satoshi's genesis block public key.")
print(f"  Without knowing the private key.")
print(f"")
print(f"  But z = {hex(z)[:24]}...")
print(f"  What message has this SHA256 hash? None.")
print(f"  It's not a hash of anything. It's algebraic garbage.")
print(f"")
print(f"  This is like showing a valid boarding pass")
print(f"  for a flight that doesn't exist.")

---

## Parte 3: Por Qué Esto No Es una Firma Real

### El problema de z

In real ECDSA, $z$ is **determined by the message**:

$$z = \text{SHA256}(\text{message})$$

The signer cannot choose $z$ — it's dictated by whatever they're signing.
The verifier independently computes $z$ from the same message and checks.

In the forgery, $z$ was derived backwards from $u$ and $s$. There is no
message. If you ask "what message was signed?", the answer is: none.
You'd have to find a message whose SHA256 hash equals that $z$ — which is
a **preimage attack on SHA256**, believed to be impossible.

### Dónde realmente reside la seguridad

```
ECDSA security chain:

  Message  ──SHA256──→  z  ──ECDSA──→  (r, s)
     ↑                  ↑                 ↑
  Chosen by          Determined by     Requires private
  the protocol       the hash           key to produce
  (not the signer)   (not the signer)   for a GIVEN z

The nonsense trick breaks the chain at z:

  ???  ──???──→  z  ←──DERIVED FROM──  (r, s)
                 ↑
              Chosen by the ATTACKER
              (security collapses)
```

### Las señales de alerta en el tweet original

1. Custom verification software (not a standard Bitcoin wallet)
2. The "message" was a raw hex hash, not a human-readable string
3. No challenge-response — the "signer" chose everything

In [ ]:
# Let's make it even more obvious: forge 5 signatures in a row
# for the same public key, each "valid", each meaningless

print("=== Forging 5 'valid' signatures for Satoshi's key ===")
print(f"(All without knowing the private key)\n")

for i in range(5):
    u_i = secrets.randbelow(SECP_N - 1) + 1
    v_i = secrets.randbelow(SECP_N - 1) + 1
    R_i = point_add(scalar_mult(u_i, G), scalar_mult(v_i, P))
    r_i = R_i.x % SECP_N
    s_i = (r_i * pow(v_i, SECP_N - 2, SECP_N)) % SECP_N
    z_i = (u_i * s_i) % SECP_N
    valid_i = ecdsa_verify_z(z_i, r_i, s_i, P)
    print(f"  Signature {i+1}: r={hex(r_i)[:14]}... s={hex(s_i)[:14]}... z={hex(z_i)[:14]}... valid={valid_i}")

print(f"\nAll 5 verify. All 5 are worthless.")
print(f"None correspond to a real message.")

In [ ]:
# Now contrast with a REAL signature: the signer can't choose z

print("=== What a REAL proof looks like ===")
print()

# Jimmy Song's challenge message from the article
challenge = "The Times 19/11/2018 The ups and downs of Downing Street"
z_real = int.from_bytes(hashlib.sha256(challenge.encode()).digest(), 'big')

print(f"Challenge: '{challenge}'")
print(f"z = SHA256(challenge) = {hex(z_real)[:24]}...")
print()
print(f"To produce a valid (r, s) for THIS specific z,")
print(f"you MUST know the private key. No trick gets around it.")
print()
print(f"The attacker's only options:")
print(f"  1. Know the private key d              → can sign anything")
print(f"  2. Find k such that (kG).x = r for     → discrete log problem")
print(f"     a specific r that satisfies the      → computationally infeasible")
print(f"     equation with this z")
print(f"  3. Find a different message with the    → SHA256 preimage attack")
print(f"     same z                               → computationally infeasible")
print()
print(f"When z is fixed by the verifier, there is no shortcut.")

---

## Parte 4: Comparación — Dos Formas de Falsificar a Satoshi

| | **2016: Blog Post** (Wright) | **2018: Tweet** (Faketoshi) |
|---|---|---|
| **Technique** | Replay a real blockchain signature | Forge a nonsense signature algebraically |
| **Requires** | A real Satoshi tx from the blockchain | Just the public key (also public) |
| **Math** | `SHA256("Sartre file") = SHA256(SHA256(modtx))` | Choose `u,v` → derive `r,s,z` backwards |
| **The z** | Real (from a 2009 transaction) | Garbage (not a hash of any message) |
| **Signature** | Real (copied from blockchain) | Fake (engineered to pass verification) |
| **Sophistication** | Moderate (needs understanding of Bitcoin tx internals) | Low (undergrad algebra) |
| **ECDSA lesson** | Verification doesn't prove *who* signed | Verification doesn't work if signer controls *z* |
| **Defense** | Verifier picks a fresh challenge message | *z* must be a hash of a real, verifier-chosen message |

### Ambos ataques fallan contra la misma defensa

```
Verifier says: "Sign this exact message: [unpredictable challenge]"

  → Replay attacker: can't, old signatures have wrong z
  → Nonsense attacker: can't, z is determined by the challenge, not by them
  → Real key holder: trivially produces valid (r, s) for any z
```

---

## Puntos Clave

1. **ECDSA is not broken.** Both tricks exploit the *protocol around* ECDSA, not the math itself.

2. **The message hash z is sacred.** If the signer chooses z, security vanishes. The verifier must determine z (by choosing the message or by both agreeing on it in advance).

3. **"Valid signature" is necessary but not sufficient.** Verification answers: "does this (r,s) satisfy the equation for this z and P?" It does NOT answer: "did the private key holder produce this for a meaningful message?"

4. **Always use standard tools.** Custom verification software is a red flag. If someone can't use a standard Bitcoin wallet to verify, ask why.

---

*Sources:*
- *[Faketoshi's Nonsense Signature](https://jimmysong.medium.com/faketoshis-nonsense-signature-8700a44536b5) — Jimmy Song*
- *[Pieter Wuille on signer-chosen message hashes](https://twitter.com/pwuille/status/1063582706288586752)*
- *[Forgery code](https://0bin.net/paste/7-tnWL-IgsKqGFcS) — Andy Poelstra, Greg Maxwell, Pieter Wuille*
- *Jimmy Song, [Programming Bitcoin](https://programmingbitcoin.com/) — Chapters 1-4*